# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.** My lane (Lane 2 — Refresh / Content Opportunity Scoring) is a "which ones first?" problem: a reviewer has limited time and thousands of candidate pages. The output they need is an ordered list — top 50, ranked by priority — not a single yes/no per page and not a set of clusters. Under the hood I'll estimate a probability that a page is declining (a classification-shaped estimate), but the thing that gets used is the ranking produced by sorting on that score, and the thing that gets graded is Precision@K on that ranking — which is exactly the scoring/ranking row in the framing-ml-problems skill's task-type table.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jamal-ai-dev/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks -> repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

# Section 1 has no required computation — it's a framing decision.
task_type = "Ranking / Scoring"
print(f"Task type: {task_type}")
print("Output needed: an ordered list of content pages, ranked by review priority.")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Starter data found. You're ready.
Task type: Ranking / Scoring
Output needed: an ordered list of content pages, ranked by review priority.


## 2. Target or proxy

**Target (proxy): `is_declining_label`** — 1 when `trend_direction == "down"`, else 0. Per the data dictionary, `trend_direction` itself comes from an observed quantity (`impressions_last_30d` vs `impressions_prev_30d`, thresholded at ±20%), not a rule I invented — so this satisfies the skill's "target must be observed, not defined" test. It is still a proxy, though, and I want to be honest about that: a declining-impressions label is not the same thing as "this page needs review" — it's the closest observable stand-in I have without running an actual editorial intervention and measuring the result. `trend_direction` and `trend_pct` are the label source only — per the flyrank-data skill, they are never used as model features.

In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

counts = df["is_declining_label"].value_counts()
pcts = df["is_declining_label"].value_counts(normalize=True).mul(100).round(1)
print("is_declining_label counts (full dataset):")
print(counts)
print()
print("is_declining_label share (%):")
print(pcts)

is_declining_label counts (full dataset):
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

is_declining_label share (%):
is_declining_label
1    54.2
0    45.8
Name: proportion, dtype: float64


## 3. Success metric

**Precision@50.** The reviewer only opens a fixed number of pages (I'm using 50, matching the starter pipeline's own evaluation), so what matters is: of the top 50 pages my score ranks first, how many are actually declining? Overall accuracy across all ~30,000 pages is the wrong metric — it rewards being right about the (many) obviously-fine pages the reviewer was never going to look at anyway. Precision@50 is the number I can defend to a reviewer directly: "of the 50 pages you're about to spend your morning on, X actually needed it." I'll also report it against a random-order baseline, so "good" has a real floor to beat, not just an absolute-sounding number.

In [8]:
def precision_at_k(is_relevant, scores, k=50):
    """is_relevant: 0/1 array; scores: higher = higher priority. Returns precision in top-k."""
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return pd.Series(is_relevant).loc[order].mean()

import numpy as np
rng = np.random.default_rng(42)
random_scores = rng.random(len(df))
random_p50 = precision_at_k(df["is_declining_label"].values, random_scores, k=50)
base_rate = df["is_declining_label"].mean()

print(f"Base rate of declining pages (full dataset): {base_rate:.3f}")
print(f"Precision@50 with a RANDOM ranking: {random_p50:.3f}")
print("Any real scoring approach needs to clear this random floor by a wide margin to be worth deploying.")

Base rate of declining pages (full dataset): 0.542
Precision@50 with a RANDOM ranking: 0.440
Any real scoring approach needs to clear this random floor by a wide margin to be worth deploying.


## 4. The unit of analysis, as a real dataframe

**One row = one content page**, aggregated over a trailing 90-day window (matching the unit of analysis I named in ML-02). The lane's working slice narrows the full 30,000-page dataset down to pages with real reviewer-relevant traffic: `impressions_90d >= 500`. Below is that slice as an actual dataframe, with the target column attached.

In [9]:
lane_df = df[df["impressions_90d"] >= 500].copy()

print(f"Full dataset: {df.shape[0]:,} rows (one row = one content page)")
print(f"Lane 2 slice (impressions_90d >= 500): {lane_df.shape[0]:,} rows")
print()

cols_to_show = ["content_id", "impressions_90d", "ctr", "avg_position",
                "trend_direction", "trend_pct", "is_declining_label"]
lane_df[cols_to_show].head(10)

Full dataset: 30,000 rows (one row = one content page)
Lane 2 slice (impressions_90d >= 500): 16,726 rows



,content_id,impressions_90d,ctr,avg_position,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,3803,0.76,10.6,down,-41.4,1
1,content_a1fb4e703a9e,15320,0.05,20.3,down,-57.7,1
2,content_9aa793d4d895,12581,0.09,36.5,down,-60.9,1
3,content_331d6c4de07b,11751,0.49,6.2,stable,-13.8,0
4,content_d99b7a2d90ca,19140,0.13,44.0,down,-34.7,1
5,content_d4084a4bc775,3970,0.03,8.5,down,-38.9,1
7,content_a63219c6e95a,1724,0.06,21.2,stable,0.6,0
8,content_5e6c160719bc,32574,0.09,46.0,down,-58.8,1
9,content_c27558df2b0c,1240,0.16,4.9,down,-29.2,1
10,content_d8ee6cc6d642,20919,1.55,2.2,stable,19.0,0


## 5. Why ML beats a fixed rule here

If one signal cleanly separated declining pages from stable ones, a single if-statement threshold would do the job and I wouldn't need a model at all. It doesn't. Below, I check the correlation between `is_declining_label` and several plausible single-signal rules a human might reach for first (search volume, CTR, position, engagement, word count). Every one is weak — none comes close to being a usable standalone rule. This echoes the Week 1 discovery that `search_volume` barely predicts `impressions_90d` (correlation ≈ 0.001): the signals that would make a simple rule work just aren't there one at a time. A model that can weigh many weak, tangled signals together is why ML earns its place here, not a hand-written threshold on any single column.

In [10]:
signal_cols = ["search_volume", "ctr", "avg_position", "engagement_rate", "word_count"]
corr = lane_df[signal_cols].corrwith(lane_df["is_declining_label"]).round(3)

print("Correlation of individual signals with is_declining_label (lane slice):")
print(corr)
print()
print("Strongest single signal:", corr.abs().idxmax(), "at", corr.abs().max())
print("Even the strongest single signal is weak — no if-statement on one column captures this.")

Correlation of individual signals with is_declining_label (lane slice):
search_volume     -0.028
ctr               -0.105
avg_position      -0.051
engagement_rate   -0.025
word_count        -0.001
dtype: float64

Strongest single signal: ctr at 0.105
Even the strongest single signal is weak — no if-statement on one column captures this.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.